# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Selected Lane:** **Lane 2: Refresh / Content Opportunity Scoring** (Provisional Predefined Lane).

**Why this lane:**
In enterprise content operations, editorial and SEO teams face an acute triage bottleneck: thousands of published pages experience organic search decay, but human editorial bandwidth is strictly finite (typically 20–50 in-depth content refreshes per monthly sprint). The starter dataset directly mirrors this reality—over 54% of the 30,000 pages show a downward trajectory (`trend_direction == 'down'`), yet search demand across these pages spans five orders of magnitude (from single-digit impressions to over 500,000 impressions). Without principled prioritization, teams either refresh pages arbitrarily, rely on brittle hand-crafted heuristics (such as sorting solely by age or raw traffic drops), or waste dozens of expensive editorial hours updating zero-demand pages while flagship high-exposure assets silently decay.

Lane 2 directly addresses this core business challenge by building a ranked opportunity queue with interpretable reason codes. It leverages observable multi-signal interactions across visibility, ranking stability, query intent, engagement, and freshness. Furthermore, Lane 2 provides the cleanest, most robust bridge across the internship: beginning with the 30,000-row starter playground (`data/raw/content_refresh_anonymized.csv`), moving to leakage-safe forward-window validation in the ~79M-row warehouse release (`dim_content` + `fact_content_daily_performance`), and culminating in an actionable, explainable decision-support playbook for real editors.

**Unit of Analysis:**
The individual pseudonymized content page / URL (`content_id`, evaluated at a fixed trailing 90-day aggregation window, associated with client context `client_id`).


In [1]:
# Verify starter data existence and inspect high-level inventory distribution across clients
import os
import pandas as pd
import numpy as np

# Support execution from both repo root and work/notebooks/
DATA_PATH = None
for candidate in ["data/raw/content_refresh_anonymized.csv", "../../data/raw/content_refresh_anonymized.csv"]:
    if os.path.exists(candidate):
        DATA_PATH = candidate
        break

if DATA_PATH is None:
    raise FileNotFoundError("Starter dataset not found at expected paths.")

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns across {df['client_id'].nunique()} unique clients.")
print("\nInventory breakdown by trend_direction:")
for trend, count in df["trend_direction"].value_counts().items():
    print(f"  - {trend:<7s}: {count:5,d} pages ({count / len(df) * 100:5.2f}%)")


Loaded dataset: 30,000 rows x 44 columns across 32 unique clients.

Inventory breakdown by trend_direction:
  - down   : 16,262 pages (54.21%)
  - stable : 5,962 pages (19.87%)
  - up     : 4,388 pages (14.63%)
  - new    : 2,236 pages ( 7.45%)
  - flat   : 1,152 pages ( 3.84%)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### 1. The Research Question
*"Which declining, high-visibility content pages should an editorial and SEO team prioritize for content refresh, expansion, or structural review to maximize recovered organic search traffic within a constrained monthly review budget?"*

### 2. What decision does this improve?
It improves the **triage and resource-allocation decision**: deciding which specific URLs from an inventory of tens of thousands should enter the active editorial sprint for investigation and updating, versus which pages should be left alone, pruned, consolidated, or merely monitored.

### 3. Who acts on the output, and what do they do (Recommended Action)?
- **Primary Actor:** Content Strategists, SEO Specialists, and Editorial Writers.
- **Recommended Action:** Reviewers inspect the top of the ranked queue (e.g., Top 50 pages) along with each page's primary reason codes (e.g., `model_decline_risk`, `stale_visible_page`, `page_one_decay_risk`, `low_ctr_visible_page`). For each prioritized page, the editor executes a targeted workflow: audits the current search engine results page (SERP) for changed searcher intent, updates outdated factual sections, enriches thin sections, refreshes meta descriptions/titles for CTR recapture, or resolves keyword cannibalization.

### 4. What does a wrong recommendation cost?
- **Cost of a False Positive (ranking an unpromising or dead page in the Top 50):**
  Wasted editorial capacity. An in-depth content overhaul requires 4 to 8 hours of skilled human labor ($300–$600 per article). If the system recommends a page with zero remaining search demand, permanent seasonal drop-off, or unrecoverable technical intent mismatch, those hours are irrecoverably lost while genuinely salvageable pages sit unattended.
- **Cost of a False Negative (failing to prioritize a decaying high-value page):**
  Compound loss of organic traffic and revenue. If a flagship revenue-driving URL slips from position 3 to page 2 (positions 11–20) due to content decay, CTR drops by 80–90%, directly reducing conversions. Regaining lost page-1 authority after competitor consolidation is substantially more expensive than proactive maintenance.

### 5. Why does data / ML help at all?
A simple if-statement (e.g., `days_since_last_update > 180`) fails because age alone does not cause traffic loss—in fact, median staleness is identical (20 days) between declining and non-declining pages. Sorting purely by raw impressions fails because high impressions often reflect vanity rankings or low-intent informational queries without immediate decay risk. Machine learning earns its place by modeling the non-linear interaction of multi-dimensional signals—combining impression depth, rank stability (`avg_position`), CTR relative to position tier, engagement rates, content length, and intent archetypes—to produce an optimal triage ranking (Precision@K) that fixed human rules cannot match.

### 6. The One-Paragraph Frame
> For Content Strategists and SEO Editors deciding which decaying pages to review first, we will build a ranked opportunity scoring model from trailing 90-day search, content, and engagement signals, scoring decay and refresh urgency measured by Precision@50 evaluated on held-out clients. A wrong call costs wasted human editorial hours on zero-demand pages or undetected organic traffic erosion on core revenue assets. A plain rule is insufficient because staleness alone does not predict decay, and single-metric heuristics ignore multi-signal interactions across visibility, rank tier, and engagement. We will claim strictly observed, directional, and decision-support results.


In [2]:
# Evidence of why a simple rule fails: Compare staleness across trend directions
# and evaluate how many pages a naive 'stale & visible' rule captures.
staleness_stats = df.groupby("trend_direction")["days_since_last_update"].agg(["median", "mean", "count"])
print("Staleness metrics by trend_direction:")
print(staleness_stats.to_string())

stale_visible_count = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"\nPages flagged by naive rule (>= 180 days stale & >= 500 impressions): {stale_visible_count} of {len(df):,} ({stale_visible_count / len(df) * 100:.2f}%)")
print("Conclusion: Naive staleness flags only 17 pages, missing thousands of high-visibility decaying pages where updates occurred recently.")


Staleness metrics by trend_direction:
                 median       mean  count
trend_direction                          
down               20.0  49.245788  16262
flat               20.0  37.453993   1152
new                20.0  22.337209   2236
stable             22.0  50.061892   5962
up                 22.0  43.425706   4388

Pages flagged by naive rule (>= 180 days stale & >= 500 impressions): 17 of 30,000 (0.06%)
Conclusion: Naive staleness flags only 17 pages, missing thousands of high-visibility decaying pages where updates occurred recently.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Analysis of the 30,000-page starter dataset (`data/raw/content_refresh_anonymized.csv`) reveals three compelling, genuine numbers that justify prioritizing Lane 2:

1. **Number 1 — The Scope of the Triage Problem (54.21% Declining Inventory):**
   Exactly **16,262 out of 30,000 pages (54.21%)** show a downward trajectory (`trend_direction == 'down'`). If an editorial team can realistically review only 50 pages per monthly sprint, that capacity addresses just **0.31%** (less than one-third of 1%) of all declining pages. An unguided team cannot possibly review them all—prioritization is strictly necessary.
2. **Number 2 — Severe Traffic Skew and Concentration (Top 50 holds 9,570,161 impressions / 11.96% of all declining demand):**
   Search exposure across declining pages is radically concentrated. While median 90-day impressions among declining pages is 961, the top 50 declining pages alone concentrate **9,570,161 impressions**—accounting for **11.96% of all 79,994,363 impressions** across the entire 16,262-page declining inventory. An effective ranking model that puts the right 50 pages in front of human editors directly safeguards millions of impressions.
3. **Number 3 — Substantial High-Demand Opportunity Surface (9,961 pages with >= 500 impressions):**
   Among the 16,262 declining pages, **9,961 pages (61.25%)** maintain at least 500 impressions over 90 days, representing meaningful searcher demand. Conversely, 3,110 declining pages have fewer than 100 impressions. A naive, unranked review risks squandering limited writer hours on low-exposure pages, whereas multi-signal opportunity scoring ensures reviews focus exclusively on substantial demand.


In [3]:
# Compute genuine supporting numbers from data/raw/content_refresh_anonymized.csv
down_pages = df[df["trend_direction"] == "down"]
n_total = len(df)
n_down = len(down_pages)
pct_down = (n_down / n_total) * 100

review_budget = 50
capacity_share = (review_budget / n_down) * 100

total_down_impressions = down_pages["impressions_90d"].sum()
top_50_impressions = down_pages["impressions_90d"].nlargest(review_budget).sum()
top_50_share = (top_50_impressions / total_down_impressions) * 100

high_demand_down = (down_pages["impressions_90d"] >= 500).sum()
high_demand_share = (high_demand_down / n_down) * 100

low_demand_down = (down_pages["impressions_90d"] < 100).sum()
low_demand_share = (low_demand_down / n_down) * 100

print(f"Number 1: {n_down:,} of {n_total:,} pages ({pct_down:.2f}%) are declining.")
print(f"          A monthly review budget of {review_budget} pages covers only {capacity_share:.2f}% of declining inventory.")
print(f"\nNumber 2: Top {review_budget} declining pages account for {top_50_impressions:,.0f} impressions")
print(f"          out of {total_down_impressions:,.0f} total declining impressions ({top_50_share:.2f}% concentration).")
print(f"\nNumber 3: {high_demand_down:,} declining pages ({high_demand_share:.2f}%) have substantial visibility (>= 500 impressions),")
print(f"          while {low_demand_down:,} declining pages ({low_demand_share:.2f}%) have negligible demand (< 100 impressions).")


Number 1: 16,262 of 30,000 pages (54.21%) are declining.
          A monthly review budget of 50 pages covers only 0.31% of declining inventory.

Number 2: Top 50 declining pages account for 9,570,161 impressions
          out of 79,994,363 total declining impressions (11.96% concentration).

Number 3: 9,961 declining pages (61.25%) have substantial visibility (>= 500 impressions),
          while 3,110 declining pages (19.12%) have negligible demand (< 100 impressions).


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

Following the data safety guidelines in `GUIDE.md` and the claims standards in `skills/framing-ml-problems/SKILL.md`:

### What this work CAN claim:
1. **Decision-Support Prioritization:** The model produces a ranked prioritization queue designed to help human editors triage pages under severe capacity constraints, putting the most promising candidates first.
2. **Observed Empirical Associations:** All findings describe measured historical relationships between observable content/search features and trailing-window traffic trajectories across 30,000 pseudonymized URLs.
3. **Directional Lift Over Baselines:** On client-holdout validation splits (testing on clients entirely unseen during training), the learned ranking demonstrates measurable empirical lift in Precision@K over fixed hand-crafted heuristic rules.
4. **Transparent Reason Codes:** Each recommendation is accompanied by observable indicators (e.g. `page_one_decay_risk`, `stale_visible_page`, `low_ctr_visible_page`) so editors know *why* a page was ranked high.

### What this work CANNOT and WILL NEVER claim:
1. **No Causal Recovery Proof:** We cannot claim that updating a page *causes* it to regain traffic, or that lack of update caused the decline. Proving causality requires randomized controlled interventions or synthetic control designs outside the scope of observational logs.
2. **No "Reverse-Engineering Google's Algorithm":** We never claim to have uncovered Google's internal ranking weights or secrets. Google evaluates billions of web pages with dynamic, confidential systems; we only observe client-side telemetry.
3. **No Product Score Replication / Circular Discovery:** We do not claim to discover internal FlyRank product flags (`health_score`, `priority_score`, etc.), nor do we use product outputs as training features.
4. **No Future Guarantees from Proxy Labels:** On the starter dataset, the label `trend_direction == 'down'` is a trailing-window proxy. Claims must explicitly acknowledge this proxy status until validated against forward-window outcomes in the full warehouse.


In [4]:
# Sanity check: Ensure strict leakage avoidance and inspect feature scale integrity
FORBIDDEN_LEAKAGE_COLS = ["trend_direction", "trend_pct", "is_declining_label"]
dataset_cols = set(df.columns)
leakage_present = [col for col in FORBIDDEN_LEAKAGE_COLS if col in dataset_cols]

print(f"Leakage check: {len(leakage_present)} target-derivation columns identified: {leakage_present}")
print("-> These columns MUST be strictly excluded from all model feature sets.\n")

print("Rate column scale verification (confirming x100 percentage representation):")
for rate_col in ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]:
    if rate_col in df.columns:
        print(f"  - {rate_col:<16s}: min={df[rate_col].min():.2f}, median={df[rate_col].median():.2f}, max={df[rate_col].max():.2f}")
print("Confirmed: Rates are scaled as percentages (e.g. median CTR ~0.35%).")


Leakage check: 2 target-derivation columns identified: ['trend_direction', 'trend_pct']
-> These columns MUST be strictly excluded from all model feature sets.

Rate column scale verification (confirming x100 percentage representation):
  - ctr             : min=0.00, median=0.07, max=100.00
  - engagement_rate : min=0.00, median=0.00, max=100.00
  - scroll_rate     : min=0.00, median=5.00, max=300.00
  - ai_traffic_pct  : min=0.00, median=0.00, max=300.00
Confirmed: Rates are scaled as percentages (e.g. median CTR ~0.35%).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
